# 37 · A provider-neutral adapter
Level L4 · Part 5 · professional · ~40 min · builds on 23, 35

## Goal
Write a thin adapter per backend so Scout's lesson 35 loop runs **unchanged** on a scripted fake, on
Gemini and on a local model (Ollama), pick the backend with one config line, and compare the three
runs on the same question.

## The idea
Scout's loop already talks to the model through one call: `model(messages, tools) -> reply`, with
messages in Scout's own **provider-neutral** shape (lesson 35). An **adapter** is the small class behind
that call: it turns neutral messages into one vendor's request, sends it, and turns the vendor's reply
back into a neutral reply. Lesson 23 wrote the request half by hand; here we add the reply half and
plug three adapters into the same loop.

The loop never imports a vendor SDK. That is the defence against **vendor lock-in**: when a model is
retired, gets pricier or goes down, you swap one adapter, not the agent.

Analogy: a travel plug adapter. Your laptop's plug never changes; you carry a small adapter per
country, and the laptop does not know which wall it is plugged into.

## Picture
```mermaid
flowchart LR
    L["run_agent<br/>(lesson 35, unchanged)"] -- "neutral messages + schemas" --> C{"CONFIG backend"}
    C --> F["FakeAdapter<br/>(scripted)"]
    C --> G["GeminiAdapter<br/>generate_content"]
    C --> O["OllamaAdapter<br/>granite4.1:3b"]
    F & G & O -- "neutral reply<br/>text or tool_calls + usage" --> L
```

## Step by step
### 1. Start the recorder
Gemini and Ollama calls are both HTTP calls, so `llm_replay` records them once to
`../data/llm_cassettes/37_provider-adapter.yaml` and replays them after that. Replay needs no key and
no Ollama server.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("37_provider-adapter")

LLM mode: replay (37_provider-adapter.yaml)


### 2. Setup from lesson 35 (shortened)
The notes, Scout's folder and the three tools, shortened as in lesson 36: `search_notes` has no pages
and the schemas are written by hand.

In [2]:
import ast, json, operator, os
from pathlib import Path

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
SCOUT_DIR = Path("../data/scout_files")
SCOUT_DIR.mkdir(parents=True, exist_ok=True)
def clear_folder():
    for old in SCOUT_DIR.glob("*.txt"):
        old.unlink()

def search_notes(query: str) -> list[dict]:
    words = query.lower().split()
    return [n for n in NOTES if all(w in n["text"].lower() for w in words)]

print(search_notes("day on Mars"))

[{'id': 3, 'text': 'A day on Mars lasts about 24 hours and 39 minutes.'}]


The calculator walks the syntax tree with an allow-list (no `eval`), and `write_file` cannot leave
Scout's folder.

In [3]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    raise ValueError("use numbers, + - * / and brackets, e.g. '24*60+39'")

def calculator(expression: str) -> float:
    return evaluate(ast.parse(expression, mode="eval").body)

def write_file(filename: str, text: str, overwrite: bool = False) -> str:
    target = (SCOUT_DIR / filename).resolve()
    if target.parent != SCOUT_DIR.resolve():
        raise ValueError(f"'{filename}' is outside Scout's folder; use a plain file name")
    if target.exists() and not overwrite:
        raise FileExistsError(f"'{filename}' exists; pick a new name or set overwrite to true")
    target.write_text(text)
    return f"saved {len(text)} characters to {target.name}"

print(calculator("24*60+39"))

1479


The schemas and lesson 35's `run_tool`, which never raises.

In [4]:
def schema(name, description, **props):
    return {"name": name, "description": description, "parameters": {
        "type": "object", "properties": {k: {"type": t} for k, t in props.items()},
        "required": [k for k in props if k != "overwrite"]}}

SCHEMAS = [schema("search_notes", "Search the user's notes for a word or phrase. Returns matching notes.",
                  query="string"),
           schema("calculator", "Evaluate arithmetic like '24*60+39'. Use it for every calculation.",
                  expression="string"),
           schema("write_file", "Save text to a file in Scout's folder.",
                  filename="string", text="string", overwrite="boolean")]
TOOLS = {"search_notes": search_notes, "calculator": calculator, "write_file": write_file}

def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available: {', '.join(tools)}", "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

Lesson 35's loop, copied **without a single change**. We take its fingerprint (lesson 35's exit test)
now and check it again at the end.

In [5]:
import hashlib, inspect

def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 8) -> dict:
    messages, errors = [{"role": "user", "text": question}], 0
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            print(f"step {step}: answer")
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step,
                    "errors": errors, "messages": messages}
        for call in calls:
            result = run_tool(call, tools)
            errors += result["is_error"]
            print(f"step {step}: {call['name']}({call['args']}) -> "
                  f"{'ERROR ' if result['is_error'] else ''}{result['content'][:70]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "errors": errors, "messages": messages}

def fingerprint(*parts) -> str:
    return hashlib.sha256("".join(inspect.getsource(p) for p in parts).encode()).hexdigest()[:12]
LOOP_BEFORE = fingerprint(run_agent, run_tool)
print("loop fingerprint:", LOOP_BEFORE)

loop fingerprint: 7d58f4cff4a4


### 3. The contract every adapter must keep
Write the interface down before writing adapters. An adapter is any callable `model(messages, tools)`
that returns a neutral reply: `{"role": "assistant", "text": ...}` or `{"role": "assistant",
"tool_calls": [{"id", "name", "args"}]}`. Every call id must be a string and `args` a plain dict.
`check_reply` enforces this, so a broken adapter fails here, not three steps into a run.

In [6]:
from typing import Protocol

class ChatModel(Protocol):
    name: str
    usage: list[dict]                 # one {"input": int, "output": int} per call
    def __call__(self, messages: list[dict], tools: list[dict]) -> dict: ...

def check_reply(reply: dict) -> dict:
    assert reply.get("role") == "assistant", f"role must be 'assistant', got {reply.get('role')!r}"
    calls = reply.get("tool_calls", [])
    assert calls or isinstance(reply.get("text"), str), "a reply needs text or tool_calls"
    for c in calls:
        assert isinstance(c.get("id"), str) and c["id"], f"tool call without an id: {c}"
        assert isinstance(c.get("args"), dict), f"args must be a dict, got {type(c.get('args')).__name__}"
    return reply

print(check_reply({"role": "assistant", "tool_calls": [{"id": "c1", "name": "calculator", "args": {}}]}))

{'role': 'assistant', 'tool_calls': [{'id': 'c1', 'name': 'calculator', 'args': {}}]}


`Protocol` is Python's way to name a shape without a base class: anything with `name`, `usage` and a
matching `__call__` counts as a `ChatModel`. The loop needs only `__call__`; `name` and `usage` are
for the comparison at the end.

### 4. Adapter 1: the scripted fake
Lesson 35's `ScriptedModel`, now with `name` and `usage`. A fake has no tokens, so usage is zero.

In [7]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class FakeAdapter:
    name = "fake"
    def __init__(self, script: list):
        self.script, self.usage, self.next_id = script, [], 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.usage.append({"input": 0, "output": 0})
        reply = self.script[min(len(self.usage), len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return check_reply({"role": "assistant", **reply})

The question and lesson 35's happy-path script: search, calculate, save, answer.

In [8]:
QUESTION = "How many minutes long is a day on Mars? Save the answer to mars_day.txt."
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "Use the calculator for arithmetic. If a tool returns an error, fix the call and try again. "
          "Answer in one sentence.")
def last_result(messages): return json.loads(messages[-1]["content"])

HAPPY = [{"tool_calls": [{"name": "search_notes", "args": {"query": "day on Mars"}}]},
         {"tool_calls": [{"name": "calculator", "args": {"expression": "24*60+39"}}]},
         lambda m: {"tool_calls": [{"name": "write_file", "args": {
             "filename": "mars_day.txt", "text": f"A day on Mars lasts about {last_result(m)} minutes."}}]},
         {"text": "A day on Mars lasts about 1479 minutes. Saved to mars_day.txt."}]
print(len(HAPPY), "scripted replies")

4 scripted replies


### 5. Adapter 2: Gemini
Lesson 35's `GeminiModel`, reorganised into the two halves of an adapter. **Request half**:
neutral messages become `types.Content`; the system prompt goes in `system_instruction`; tool results
go back as `function_response` parts with the call's id.

Gemini's newer Interactions API is not reachable on Vertex express mode with this key (it returns 404),
so this adapter uses `generate_content`, as lessons 22 and 35 did. Only this adapter would change if you
moved to Interactions; the loop would not.

In [9]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")

def to_gemini(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                  # Gemini's own turn, thought signature included
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(id=m["call_id"], name=m["name"],
                                                                       response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)        # parallel results share one user turn
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

**Reply half**: read `function_calls` and usage, and return a neutral reply. The vendor's own turn is
kept under `raw`, because Gemini needs it back unchanged (its thought signature, lessons 22 and 35).
Only this adapter ever reads `raw`.

In [10]:
class GeminiAdapter:
    name = "gemini"
    def __init__(self, model: str = MODEL, system: str = SYSTEM):
        self.model, self.system, self.usage = model, system, []

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        decls = [types.FunctionDeclaration(name=s["name"], description=s["description"],
                                           parameters_json_schema=s["parameters"]) for s in tools]
        cfg = types.GenerateContentConfig(system_instruction=self.system, max_output_tokens=400,
            tools=[types.Tool(function_declarations=decls)],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
        resp = client.models.generate_content(model=self.model, contents=to_gemini(messages), config=cfg)
        u = resp.usage_metadata
        self.usage.append({"input": u.prompt_token_count, "output": u.candidates_token_count or 0})
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        reply = {"tool_calls": calls} if calls else {"text": resp.text or ""}
        return check_reply({"role": "assistant", **reply, "raw": raw})

### 6. Adapter 3: a local model on Ollama
The `ollama` client (lesson 16) speaks a third format. The system prompt is the first message, a tool
schema is wrapped in `{"type": "function", "function": ...}`, and a tool result is a `tool` message
with `tool_name`. The request half:

In [11]:
import ollama

LOCAL_MODEL = "granite4.1:3b"
local = ollama.Client(host="http://localhost:11434")      # creating a client sends nothing

def to_ollama(messages: list[dict], system: str) -> list[dict]:
    out = [{"role": "system", "content": system}]
    for m in messages:
        if m["role"] == "user":
            out.append({"role": "user", "content": m["text"]})
        elif m["role"] == "assistant":
            out.append({"role": "assistant", "content": m.get("text", ""), "tool_calls": [
                {"function": {"name": c["name"], "arguments": c["args"]}} for c in m.get("tool_calls", [])]})
        else:
            out.append({"role": "tool", "tool_name": m["name"], "content": m["content"]})
    return out

print([m["role"] for m in to_ollama([{"role": "user", "text": QUESTION}], SYSTEM)])

['system', 'user']


The reply half has a surprise: Ollama's tool calls carry **no id** (its `ToolCall` has only `function`
with `name` and `arguments`). Scout's loop pairs each result with its call id (lesson 23), so the
adapter **mints** one. Usage comes from Ollama's `prompt_eval_count` and `eval_count`.

In [12]:
class OllamaAdapter:
    name = "local"
    def __init__(self, model: str = LOCAL_MODEL, system: str = SYSTEM):
        self.model, self.system, self.usage, self.minted = model, system, [], 0

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        resp = local.chat(model=self.model, messages=to_ollama(messages, self.system),
                          tools=[{"type": "function", "function": s} for s in tools],
                          options={"temperature": 0, "num_predict": 300})
        self.usage.append({"input": resp.prompt_eval_count or 0, "output": resp.eval_count or 0})
        calls = []
        for tc in resp.message.tool_calls or []:
            self.minted += 1                                   # Ollama gives no id: make one
            calls.append({"id": f"local_{self.minted}", "name": tc.function.name,
                          "args": dict(tc.function.arguments)})
        reply = {"tool_calls": calls} if calls else {"text": resp.message.content or ""}
        return check_reply({"role": "assistant", **reply})

### 7. One config line picks the backend
A small factory maps a name to an adapter. Changing `CONFIG["backend"]` is the only edit needed to move
Scout to another model. In a real app the value would come from an environment variable or a config file.

In [13]:
CONFIG = {"backend": "fake"}                     # "fake" | "gemini" | "local"

def make_model(backend: str) -> ChatModel:
    factories = {"fake": lambda: FakeAdapter(HAPPY), "gemini": GeminiAdapter, "local": OllamaAdapter}
    if backend not in factories:
        raise ValueError(f"unknown backend {backend!r}; choose one of {sorted(factories)}")
    return factories[backend]()

def ask_scout(backend: str) -> dict:
    clear_folder()
    model = make_model(backend)
    run = run_agent(QUESTION, model, TOOLS, SCHEMAS)
    saved = SCOUT_DIR / "mars_day.txt"
    run.update(backend=backend, usage=model.usage, saved=saved.read_text() if saved.exists() else None)
    return run

Run Scout on the configured backend, the fake. Results are kept per backend for the comparison.

In [14]:
RESULTS = {}
RESULTS[CONFIG["backend"]] = ask_scout(CONFIG["backend"])
print(RESULTS["fake"]["answer"], "| saved:", RESULTS["fake"]["saved"])

step 1: search_notes({'query': 'day on Mars'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes.
step 2: calculator({'expression': '24*60+39'}) -> 1479
step 3: write_file({'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 minutes.'}) -> "saved 39 characters to mars_day.txt"
step 4: answer
A day on Mars lasts about 1479 minutes. Saved to mars_day.txt. | saved: A day on Mars lasts about 1479 minutes.


### 8. Same loop, Gemini
Only the config value changes. This cell is recorded output: it replays from the cassette.

In [15]:
CONFIG["backend"] = "gemini"
RESULTS[CONFIG["backend"]] = ask_scout(CONFIG["backend"])
print(RESULTS["gemini"]["answer"], "| saved:", RESULTS["gemini"]["saved"])

step 1: search_notes({'query': 'Mars'}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
step 2: calculator({'expression': '24*60+39'}) -> 1479
step 3: write_file({'text': '1479 minutes', 'filename': 'mars_day.txt', 'overwrite': True}) -> "saved 12 characters to mars_day.txt"
step 4: answer
A day on Mars is 1,479 minutes long, and I have saved this answer to mars_day.txt. | saved: 1479 minutes


### 9. Same loop, the local model
The 3B model runs on the laptop's CPU (recorded from a real Ollama server; replay needs neither Ollama
nor a key). With temperature 0 it is repeatable on the same machine.

In [16]:
CONFIG["backend"] = "local"
RESULTS[CONFIG["backend"]] = ask_scout(CONFIG["backend"])
print(RESULTS["local"]["answer"], "| saved:", RESULTS["local"]["saved"])

step 1: search_notes({'query': 'Mars day'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes.
step 2: calculator({'expression': '24*60 + 39'}) -> 1479
step 3: write_file({'filename': 'mars_day.txt', 'text': '1479'}) -> "saved 4 characters to mars_day.txt"
step 4: answer
A day on Mars is 24 hours and 39 minutes, which equals **1479 minutes**. This value has been saved to **mars_day.txt**. | saved: 1479


In this recording the 3B model searched for "Mars day", calculated `24*60 + 39`, saved "1479" and
answered in four steps: the same path as Gemini. Do not read too much into one run: lesson 16 saw this
model call a tool for a plain greeting and get a time calculation confidently wrong (lesson 16).

### 10. Compare the backends
One table from the neutral results. "correct" means the answer and the saved file both contain 1479
(commas removed first, so "1,479" counts). If a model cell above was skipped, its row is missing.

In [17]:
print(f"{'backend':8} {'stop':12} {'steps':>5} {'errors':>6} {'calls':>5} {'in tok':>7} {'out tok':>7}  correct")
for name, r in RESULTS.items():
    tool_calls = sum(len(m.get("tool_calls", [])) for m in r["messages"])
    tin, tout = sum(u["input"] for u in r["usage"]), sum(u["output"] for u in r["usage"])
    correct = all("1479" in (s or "").replace(",", "") for s in (r["answer"], r["saved"]))
    print(f"{name:8} {r['stop']:12} {r['steps']:>5} {r['errors']:>6} {tool_calls:>5} {tin:>7} {tout:>7}  {correct}")

backend  stop         steps errors calls  in tok out tok  correct
fake     end_of_turn      4      0     3       0       0  True
gemini   end_of_turn      4      0     3     956      98  True
local    end_of_turn      4      0     3    1667     109  True


All three took four steps, called the same three tools and made no errors. Gemini wrote "1,479";
without stripping the comma, the check would have marked a right answer wrong. Scoring needs as much
care as the run (lesson 48). The local model used about 1.7 times Gemini's input tokens for the same
conversation: each runtime wraps messages and tools in its own template, so token counts are not
comparable across vendors. Local tokens cost no money, but they cost CPU time.

And the proof the loop did not move: its fingerprint after all three backends.

In [18]:
print("loop unchanged:", fingerprint(run_agent, run_tool) == LOOP_BEFORE)
print("tool names each backend called:")
for name, r in RESULTS.items():
    print(f"  {name:7}", [c["name"] for m in r["messages"] for c in m.get("tool_calls", [])])

loop unchanged: True
tool names each backend called:
  fake    ['search_notes', 'calculator', 'write_file']
  gemini  ['search_notes', 'calculator', 'write_file']
  local   ['search_notes', 'calculator', 'write_file']


### 11. Where the abstraction leaks
A neutral format is never perfectly neutral. Three leaks in this lesson alone:
- **Gemini's `raw` turn.** A history made by Gemini carries `raw`; one made elsewhere does not. Gemini on
  Vertex rejects a rebuilt function-call turn without its thought signature (observed on Vertex:
  400 INVALID_ARGUMENT, 'Function call is missing a thought_signature'; see Google's thought-signatures
  docs), so moving a conversation *mid-way* onto Gemini needs care. Starting fresh on any backend is safe.
- **Ids.** Gemini issues them, Ollama does not, so the Ollama adapter makes its own.
- **Usage.** Each vendor counts and names tokens its own way; the adapter normalises names, not meaning.

The rule: vendor-only data may ride along in the neutral message, but only its own adapter reads it.

## At work
- **Gateways do this at company scale.** LiteLLM (see its docs) gives one call shape over 100+
  providers for gateways and fallbacks; OpenAI-compatible endpoints (Ollama, LM Studio, many
  hosts) let one client reach many backends, but tool-call fidelity varies by model. A Gemini Developer
  API key would let you write an Interactions adapter behind the same contract. (Track 6.)
- **Model ids live in config, and swaps are gated by evals.** Vendors retire models on fixed dates
  (Anthropic's model deprecations page: Haiku 4.5 not before 2026-10-15); teams keep the id in
  one config value and run an eval set before switching. → lesson 48
- **The fake adapter is the test double.** Because every backend meets one contract, the loop's unit
  tests run on the scripted fake with no key, and cassettes cover the real adapters. → lesson 46
- **A fair comparison needs many questions and repeats**, with cost and latency per backend. → lesson 45

## What just happened
- Scout's lesson 35 loop ran on three backends; its fingerprint did not change.
- Each adapter had a request half (neutral → vendor) and a reply half (vendor → neutral), and
  `check_reply` enforced one contract on every reply.
- `CONFIG["backend"]` was the only line changed between runs.
- The Ollama adapter minted call ids, because Ollama returns none; the Gemini adapter kept its `raw`
  turn for its own use.
- One table compared steps, errors, tool calls, tokens and correctness from neutral results.

## Common mistakes
**1. Letting a vendor object leak into the loop.** This loop step reads `reply.function_calls`, which
only exists on a Gemini response. It works while Gemini is the only backend, and breaks the moment you
plug in anything else.

In [19]:
def first_step_leaky(question, model, schemas):
    reply = model([{"role": "user", "text": question}], schemas)
    return [fc.name for fc in reply.function_calls or []]     # Gemini SDK attribute

first_step_leaky(QUESTION, FakeAdapter(HAPPY), SCHEMAS)

AttributeError: 'dict' object has no attribute 'function_calls'

Fix: the loop reads only neutral keys (`reply.get("tool_calls", [])`); vendor attributes stay inside
the adapter.

In [20]:
reply = FakeAdapter(HAPPY)([{"role": "user", "text": QUESTION}], SCHEMAS)
print([c["name"] for c in reply.get("tool_calls", [])])

['search_notes']


**2. Copying Ollama's tool calls through as they are.** A reply half that forgets to mint ids returns
calls without `id`. This `Message` is constructed by hand to look like Ollama's reply (it is not model
output; no server is involved), and we parse it the naive way. `check_reply` stops it before the loop does.

In [21]:
from ollama import Message
msg = Message(role="assistant", content="", tool_calls=[Message.ToolCall(
    function=Message.ToolCall.Function(name="calculator", arguments={"expression": "24*60+39"}))])
naive = {"role": "assistant", "tool_calls": [{"name": tc.function.name, "args": dict(tc.function.arguments)}
                                             for tc in msg.tool_calls]}
check_reply(naive)

AssertionError: tool call without an id: {'name': 'calculator', 'args': {'expression': '24*60+39'}}

Without `check_reply`, the loop would fail later with a bare `KeyError: 'id'` when it appends the tool
result. Fix: mint an id per call, as `OllamaAdapter` does.

In [22]:
fixed = {"role": "assistant", "tool_calls": [{"id": f"local_{i}", **c} for i, c in enumerate(naive["tool_calls"], 1)]}
print(check_reply(fixed)["tool_calls"])

[{'id': 'local_1', 'name': 'calculator', 'args': {'expression': '24*60+39'}}]


**3. A typo in the config value.** With a dict lookup and no check, `"Gemini"` fails deep inside with a
bare `KeyError`. `make_model` raises a clear error that lists the valid names.

In [23]:
make_model("Gemini")

ValueError: unknown backend 'Gemini'; choose one of ['fake', 'gemini', 'local']

Fix: use the exact lower-case name (and validate config at start-up, not on the first question).

In [24]:
print(make_model("fake").name)

fake


## Try it
Write a fourth adapter's two halves for the **OpenAI-compatible Chat Completions** format that
Ollama's `/v1`, LM Studio and many hosts speak (lesson 16). `to_chat(messages, system)` must produce
`{"role": "assistant", "tool_calls": [{"id", "type": "function", "function": {"name", "arguments"}}]}`
with `arguments` as a **JSON string**, and tool results as `{"role": "tool", "tool_call_id", "content"}`.
`from_chat(message: dict)` turns such an assistant message back into a neutral reply that passes
`check_reply`. Test both on `RESULTS["fake"]["messages"]`: convert to chat format and back, and check
the call ids and arguments survive.

In [25]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [26]:
llm_replay.stop()

## Key terms
- **adapter (provider adapter)** — a small class that implements Scout's `model(messages, tools)` call for one backend: a request half (neutral → vendor) and a reply half (vendor → neutral).
- **provider-neutral messages** — the app's own message shape (`user` / `assistant` with `text` or `tool_calls` / `tool` with `call_id`), which no vendor SDK defines and every adapter converts.
- **vendor lock-in** — when switching model provider means rewriting the app, because vendor formats or objects have spread through its code.
- **adapter contract** — the rules every adapter's reply must meet (role, text or tool calls, string ids, dict args), checked by `check_reply`.
- **minted id** — a call id the adapter makes up when the backend (here Ollama) returns tool calls without one.